In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import time
from datetime import datetime, timedelta

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error,
    precision_score, recall_score, f1_score,
    confusion_matrix, pairwise_distances,r2_score
)

import tensorflow as tf
import keras
from keras.models import Sequential
from keras.layers import SimpleRNN, Dense, Dropout, Input
from keras.callbacks import EarlyStopping, ReduceLROnPlateau

from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings('ignore')
tf.get_logger().setLevel('ERROR')
%matplotlib inline

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

In [ ]:
DATA_DIR = r"C:\Users\Hp\Desktop\Trafic_Routier\PEMS07"

npz_path = os.path.join(DATA_DIR, "PEMS07.npz")
csv_path = os.path.join(DATA_DIR, "PEMS07.csv")

if os.path.exists(npz_path):
    npz_data = np.load(npz_path)
    print(f"Clés NPZ : {list(npz_data.keys())}")
    data = npz_data['data'] if 'data' in npz_data else npz_data[list(npz_data.keys())[0]]
    if data.ndim == 3:
        print(f"Shape brute (T, N, Features): {data.shape}")
        data = data[:, :, 0]
elif os.path.exists(csv_path):
    data = pd.read_csv(csv_path, header=None).values
else:
    raise FileNotFoundError(f"Aucun fichier trouvé dans {DATA_DIR}")

T, N = data.shape

start_date = datetime(2017, 5, 1)
timestamps = [start_date + timedelta(minutes=5 * i) for i in range(T)]

df = pd.DataFrame(data, columns=[f'Capteur_{i}' for i in range(N)], index=timestamps)
df.index.name = 'Timestamp'

print(f"\nShape finale : ({T}, {N})")
print(f"→ {T} timestamps (pas de 5 min) = ~{T*5/60/24:.0f} jours")
print(f"→ {N} capteurs")
print(f"Période : {df.index[0]} → {df.index[-1]}")
df

In [ ]:
flat = data.flatten()
print(f"Nombre total de mesures : {len(flat):,} ({T} timestamps × {N} capteurs)")
print(f"\nMin       : {flat.min():.4f}")
print(f"Max       : {flat.max():.4f}")
print(f"Moyenne   : {flat.mean():.4f}")
print(f"Médiane   : {np.median(flat):.4f}")
print(f"Écart-type: {flat.std():.4f}")
print(f"\nQuantiles :")
for q in [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]:
    print(f"  Q{int(q*100):02d} = {np.quantile(flat, q):.4f}")
print(f"\nValeurs manquantes (NaN) : {np.isnan(data).sum()}")
print(f"Valeurs nulles (=0)      : {(data == 0).sum()}")

In [ ]:

sensor_means = data.mean(axis=0)
sensor_stds = data.std(axis=0)


problematic_sensors = np.where(sensor_stds < 1e-4)[0]

print(f"Bilan de santé des {N} capteurs :")
if len(problematic_sensors) > 0:
    print(f"⚠️ {len(problematic_sensors)} capteurs problématiques détectés (quasi-constants) :")
    for sid in problematic_sensors:
        print(f"   - Capteur {sid} : Moyelle={sensor_means[sid]:.4f}, Écart-type={sensor_stds[sid]:.4f}")
else:
    print("✅ Tous les capteurs présentent une variabilité normale.")


fig, ax = plt.subplots(figsize=(16, 8))
sorted_idx = np.argsort(sensor_means)
ax.barh(range(N), sensor_means[sorted_idx],
        color=plt.cm.RdYlGn(np.linspace(0.1, 0.9, N)), height=0.8)
ax.set_xlabel('Vitesse Moyenne (Z-score)')
ax.set_ylabel('Capteur (ordonné par performance)')
ax.set_title(f'Rapport de Santé : Classement des {N} Capteurs par Performance',
             fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

In [ ]:
mean_speed = data.mean(axis=1)

mean_speed_idx = (mean_speed - mean_speed.min()) / (mean_speed.max() - mean_speed.min()) * 100

fig, ax = plt.subplots(figsize=(20, 6))
ax.plot(timestamps, mean_speed_idx, lw=1.2, color='#2c3e50')


idx_max = np.argmax(mean_speed_idx)
idx_min = np.argmin(mean_speed_idx)
ax.scatter(timestamps[idx_max], mean_speed_idx[idx_max], color='green', s=100, zorder=5)
ax.annotate(f'Pic Fluidité = {mean_speed_idx[idx_max]:.1f}%',
            xy=(timestamps[idx_max], mean_speed_idx[idx_max]),
            xytext=(10, 15), textcoords='offset points',
            fontsize=10, fontweight='bold', color='green',
            arrowprops=dict(arrowstyle='->', color='green'))
ax.scatter(timestamps[idx_min], mean_speed_idx[idx_min], color='red', s=100, zorder=5)
ax.annotate(f'Congestion Max = {mean_speed_idx[idx_min]:.1f}%',
            xy=(timestamps[idx_min], mean_speed_idx[idx_min]),
            xytext=(10, -20), textcoords='offset points',
            fontsize=10, fontweight='bold', color='red',
            arrowprops=dict(arrowstyle='->', color='red'))

ax.set_xlabel('Temps', fontsize=13)
ax.set_ylabel('Indice de Fluidité (%)', fontsize=13)
ax.set_title('Traffic × Temps — Évolution de la Fluidité (0-100%)',
             fontsize=15, fontweight='bold')
ax.set_ylim(-5, 105)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 6))

q33 = np.percentile(flat, 33)
q66 = np.percentile(flat, 66)

ax = axes[0]
ax.axvspan(flat.min(), q33, alpha=0.15, color='red', label=f'Bouchon (< {q33:.2f})')
ax.axvspan(q33, q66, alpha=0.15, color='orange', label=f'Congestion ({q33:.2f}–{q66:.2f})')
ax.axvspan(q66, flat.max(), alpha=0.10, color='green', label=f'Fluide (> {q66:.2f})')

sns.kdeplot(flat, ax=ax, color='navy', lw=2.5, fill=True, alpha=0.3)
ax.set_xlabel('Vitesse (Z-score ou normalisée)', fontsize=12)
ax.set_ylabel('Densité', fontsize=12)
ax.set_title('Densité Globale des Vitesses — Classification du Trafic',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_xlim(flat.min(), np.percentile(flat, 99.5))

ax2 = axes[1]
sample_ids = np.linspace(0, N-1, 10, dtype=int)
colors_sample = plt.cm.tab10(np.linspace(0, 1, len(sample_ids)))
for idx, sid in enumerate(sample_ids):
    sns.kdeplot(data[:, sid], ax=ax2, label=f'Capteur {sid}',
                color=colors_sample[idx], lw=1.5, alpha=0.7)
ax2.set_xlabel('Vitesse (Z-score ou normalisée)', fontsize=12)
ax2.set_ylabel('Densité', fontsize=12)
ax2.set_title('Densité par Capteur (échantillon)', fontsize=13, fontweight='bold')
ax2.legend(fontsize=8, ncol=2)
ax2.set_xlim(flat.min(), np.percentile(flat, 99.5))

plt.tight_layout()
plt.show()


bouchon_pct = (flat < q33).sum() / len(flat) * 100
congestion_pct = ((flat >= q33) & (flat < q66)).sum() / len(flat) * 100
fluide_pct = (flat >= q66).sum() / len(flat) * 100
print(f"Répartition du trafic :")
print(f"  Bouchon    (< {q33:.2f})  : {bouchon_pct:.1f}%")
print(f"  Congestion ({q33:.2f} - {q66:.2f}) : {congestion_pct:.1f}%")
print(f"  Fluide     (> {q66:.2f})  : {fluide_pct:.1f}%")

In [ ]:
df['hour'] = df.index.hour

global_hourly = df.groupby('hour').mean().mean(axis=1)
global_hourly_idx = (global_hourly - global_hourly.min()) / (global_hourly.max() - global_hourly.min()) * 100

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(global_hourly_idx.index, global_hourly_idx.values,
        lw=2, color='#2c3e50', marker='o', markersize=6)

h_max = global_hourly_idx.idxmax()
h_min = global_hourly_idx.idxmin()
ax.scatter(h_max, global_hourly_idx[h_max], color='green', s=120, zorder=5)
ax.annotate(f'Pic Fluidité = {global_hourly_idx[h_max]:.1f}% ({h_max}h)',
            xy=(h_max, global_hourly_idx[h_max]), xytext=(10, 15),
            textcoords='offset points', fontsize=10, fontweight='bold', color='green',
            arrowprops=dict(arrowstyle='->', color='green'))

ax.scatter(h_min, global_hourly_idx[h_min], color='red', s=120, zorder=5)
ax.annotate(f'Max Congestion = {global_hourly_idx[h_min]:.1f}% ({h_min}h)',
            xy=(h_min, global_hourly_idx[h_min]), xytext=(10, -20),
            textcoords='offset points', fontsize=10, fontweight='bold', color='red',
            arrowprops=dict(arrowstyle='->', color='red'))

ax.set_xlabel('Heure de la Journée', fontsize=13)
ax.set_ylabel('Indice de Fluidité (%)', fontsize=13)
ax.set_title('Profil Journalier — Évolution de la Fluidité par Heure (0-100%)',
             fontsize=15, fontweight='bold')
ax.set_xticks(range(24))
ax.set_xticklabels([f'{h}h' for h in range(24)])
ax.set_ylim(-5, 120) 
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
df['dayofweek'] = df.index.dayofweek

day_names = ['Lundi', 'Mardi', 'Mercredi', 'Jeudi', 'Vendredi', 'Samedi', 'Dimanche']
global_weekly = df.drop(columns=['hour']).groupby('dayofweek').mean().mean(axis=1)

global_weekly_idx = (global_weekly - global_weekly.min()) / (global_weekly.max() - global_weekly.min()) * 100

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(global_weekly_idx.index, global_weekly_idx.values,
        lw=2.5, color='#2c3e50', marker='o', markersize=8)


d_max = global_weekly_idx.idxmax()
d_min = global_weekly_idx.idxmin()
ax.scatter(d_max, global_weekly_idx[d_max], color='green', s=120, zorder=5)
ax.annotate(f'Pic Fluidité = {global_weekly_idx[d_max]:.1f}% ({day_names[d_max]})',
            xy=(d_max, global_weekly_idx[d_max]), xytext=(10, 15),
            textcoords='offset points', fontsize=10, fontweight='bold', color='green',
            arrowprops=dict(arrowstyle='->', color='green'))

ax.scatter(d_min, global_weekly_idx[d_min], color='red', s=120, zorder=5)
ax.annotate(f'Max Congestion = {global_weekly_idx[d_min]:.1f}% ({day_names[d_min]})',
            xy=(d_min, global_weekly_idx[d_min]), xytext=(10, -20),
            textcoords='offset points', fontsize=10, fontweight='bold', color='red',
            arrowprops=dict(arrowstyle='->', color='red'))

ax.set_xlabel('Jour de la Semaine', fontsize=13)
ax.set_ylabel('Indice de Fluidité (%)', fontsize=13)
ax.set_title('Profil Hebdomadaire — Évolution de la Fluidité par Jour (0-100%)',
             fontsize=15, fontweight='bold')
ax.set_xticks(range(7))
ax.set_xticklabels(day_names)
ax.set_ylim(-5, 120)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:

corr_with_sensor_0 = np.corrcoef(data.T)[0, :]
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(range(N), corr_with_sensor_0, color='darkblue', lw=1.5)
ax.fill_between(range(N), corr_with_sensor_0, 0, where=(corr_with_sensor_0 > 0), color='green', alpha=0.3)
ax.fill_between(range(N), corr_with_sensor_0, 0, where=(corr_with_sensor_0 < 0), color='red', alpha=0.3)
ax.set_title('Profil de Corrélation Spatiale (Référence = Capteur 0)', fontsize=14, fontweight='bold')
ax.set_xlabel('Index du Capteur')
ax.set_ylabel('Coefficient de Corrélation (Pearson)')
ax.set_xlim(0, N-1)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:

df.drop(columns=['hour', 'dayofweek'], inplace=True, errors='ignore')

scaler = MinMaxScaler(feature_range=(0, 1))
data_norm = scaler.fit_transform(data)

print('Avant normalisation :')
print(f'  Min = {data.min():.4f}, Max = {data.max():.4f}, Moyenne = {data.mean():.4f}')
print(f'\nAprès normalisation :')
print(f'  Min = {data_norm.min():.4f}, Max = {data_norm.max():.4f}, Moyenne = {data_norm.mean():.4f}')

In [ ]:
SEQ_LENGTH = 12
TEST_RATIO = 0.20
CONGESTION_THRESHOLD = 0.4

split_idx = int(T * (1 - TEST_RATIO))
train_data = data_norm[:split_idx]
test_data = data_norm[split_idx:]

print(f'Train : {train_data.shape[0]} timestamps ({train_data.shape[0]/T*100:.1f}%)')
print(f'Test  : {test_data.shape[0]} timestamps ({test_data.shape[0]/T*100:.1f}%)')

def create_sequences(data, seq_len):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i+seq_len])
        y.append(data[i+seq_len])
    return np.array(X), np.array(y)

X_train, y_train = create_sequences(train_data, SEQ_LENGTH)
X_test, y_test = create_sequences(test_data, SEQ_LENGTH)

print(f'\nX_train : {X_train.shape}')
print(f'y_train : {y_train.shape}')
print(f'X_test  : {X_test.shape}')
print(f'y_test  : {y_test.shape}')

sample_sensors = np.linspace(0, N-1, 4, dtype=int)

In [ ]:
RNN_UNITS = 64
EPOCHS = 30
BATCH_SIZE = 64

model = Sequential([
    Input(shape=(SEQ_LENGTH, N)),
    SimpleRNN(RNN_UNITS, activation='tanh', return_sequences=True),
    Dropout(0.2),
    SimpleRNN(RNN_UNITS // 2),
    Dropout(0.2),
    Dense(N)
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
]
start = time.time()
history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.1,
    callbacks=callbacks,
    verbose=1
)
end = time.time()
temps_rnn = end - start
print("Temps d'execution RNN:", temps_rnn, "secondes")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(history.history['loss'], label='Train Loss', lw=2)
ax.plot(history.history['val_loss'], label='Validation Loss', lw=2)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss (MSE)')
ax.set_title('Courbes de Perte — SimpleRNN', fontsize=14, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
y_pred = model.predict(X_test, verbose=0)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = np.mean(np.abs(y_test - y_pred))
r2   = r2_score(y_test.flatten(), y_pred.flatten())

y_test_bin = (y_test < CONGESTION_THRESHOLD).astype(int).flatten()
y_pred_bin = (y_pred < CONGESTION_THRESHOLD).astype(int).flatten()

precision = precision_score(y_test_bin, y_pred_bin, average='binary', zero_division=0)
recall = recall_score(y_test_bin, y_pred_bin, average='binary', zero_division=0)
f1 = f1_score(y_test_bin, y_pred_bin, average='binary', zero_division=0)

print('╔' + '═'*45 + '╗')
print('║      RÉSULTATS DU MODÈLE SimpleRNN        ║')
print('╠' + '═'*45 + '╣')
print(f'║  MSE       : {mse:.6f}')
print(f'║  RMSE      : {rmse:.6f}')
print(f'║  MAE       : {mae:.6f}')
print(f'║  R²        : {r2:.6f}') 
print('╠' + '═'*45 + '╣')
print(f'║  Precision : {precision:.4f}')
print(f'║  Recall    : {recall:.4f}')
print(f'║  F1-Score  : {f1:.4f}')
print('╚' + '═'*45 + '╝')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, sensor in zip(axes.flatten(), sample_sensors):
    ax.plot(y_test[:200, sensor], label='Réel', lw=1.5, alpha=0.8)
    ax.plot(y_pred[:200, sensor], label='Prédit', lw=1.5, alpha=0.8)
    ax.set_title(f'Capteur {sensor}', fontweight='bold')
    ax.set_xlabel('Timestamp'); ax.set_ylabel('Vitesse (norm.)')
    ax.legend(); ax.grid(True, alpha=0.3)
plt.suptitle('Prédictions vs Réel — Échantillon de Capteurs (RNN)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(20, 7, figsize=(35, 60))
for i, ax in enumerate(axes.flatten()):
    ax.plot(y_test[:200, i], label='Réel', lw=1, alpha=0.8, color='steelblue')
    ax.plot(y_pred[:200, i], label='Prédit', lw=1, alpha=0.8, color='orange')
    ax.set_title(f'Capteur {i}', fontsize=8, fontweight='bold')
    ax.set_xlabel('Timestamp', fontsize=6); ax.set_ylabel('Vitesse', fontsize=6)
    ax.tick_params(labelsize=6); ax.grid(True, alpha=0.3)
    if i == 0: ax.legend(fontsize=6)
plt.suptitle('Prédictions vs Réel — 140 Capteurs (RNN)', fontsize=16, fontweight='bold', y=1.001)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.svm import SVR
from sklearn.multioutput import MultiOutputRegressor

def create_sequences_flat(data, seq_len):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i+seq_len].flatten())
        y.append(data[i+seq_len])
    return np.array(X), np.array(y)

X_train_svm, y_train_svm = create_sequences_flat(train_data, SEQ_LENGTH)
X_test_svm, y_test_svm = create_sequences_flat(test_data, SEQ_LENGTH)
print(f'X_train_svm: {X_train_svm.shape}')
print(f'X_test_svm : {X_test_svm.shape}')

MAX_TRAIN = 2000
X_tr = X_train_svm[:MAX_TRAIN]
y_tr = y_train_svm[:MAX_TRAIN]


In [ ]:
start = time.time()
svr_base = SVR(kernel='rbf', C=1.0, epsilon=0.05, gamma='scale')
svm_model = MultiOutputRegressor(svr_base, n_jobs=1)
svm_model.fit(X_tr, y_tr)
end = time.time()
temps_svm = end - start
print("Temps d'execution SVM :", temps_svm, "secondes")

y_pred_svm = svm_model.predict(X_test_svm)
y_true_svm = y_test_svm
print(f'y_pred_svm shape: {y_pred_svm.shape}')


In [ ]:
mse_svm = mean_squared_error(y_true_svm, y_pred_svm)
rmse_svm = np.sqrt(mse_svm)
mae_svm = mean_absolute_error(y_true_svm, y_pred_svm)
r2_svm   = r2_score(y_true_svm.flatten(), y_pred_svm.flatten())
mask = y_true_svm != 0
mape_svm = np.mean(np.abs((y_true_svm[mask]-y_pred_svm[mask])/y_true_svm[mask]))*100
y_tb_svm = (y_true_svm < CONGESTION_THRESHOLD).astype(int).flatten()
y_pb_svm = (y_pred_svm < CONGESTION_THRESHOLD).astype(int).flatten()
prec_svm = precision_score(y_tb_svm, y_pb_svm, zero_division=0)
rec_svm = recall_score(y_tb_svm, y_pb_svm, zero_division=0)
f1_svm = f1_score(y_tb_svm, y_pb_svm, zero_division=0)

print('='*50)
print('     RÉSULTATS SVM (SVR-RBF)')
print('='*50)
print(f'  MSE       : {mse_svm:.6f}')
print(f'  RMSE      : {rmse_svm:.6f}')
print(f'  MAE       : {mae_svm:.6f}')
print(f'  R²        : {r2_svm:.6f}')
print(f'  MAPE      : {mape_svm:.2f} %')
print(f'  Precision : {prec_svm:.4f}')
print(f'  Recall    : {rec_svm:.4f}')
print(f'  F1-Score  : {f1_svm:.4f}')
print('='*50)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, s in zip(axes.flatten(), sample_sensors):
    ax.plot(y_true_svm[:200, s], label='Réel', lw=1.5, color='steelblue')
    ax.plot(y_pred_svm[:200, s], label='SVM', lw=1.5, color='green')
    ax.set_title(f'Capteur {s}', fontweight='bold')
    ax.legend(); ax.grid(True, alpha=0.3)
plt.suptitle('SVM (SVR-RBF) — Prédictions vs Réel', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:

fig, axes = plt.subplots(20, 7, figsize=(35, 60))
for i, ax in enumerate(axes.flatten()):
    ax.plot(y_true_svm[:200, i], lw=1, alpha=0.8, color='steelblue', label='Réel')
    ax.plot(y_pred_svm[:200, i], lw=1, alpha=0.8, color='green', label='SVM')
    ax.set_title(f'Capteur {i}', fontsize=8, fontweight='bold')
    ax.set_xlabel('Timestamp', fontsize=6)
    ax.set_ylabel('Vitesse', fontsize=6)
    ax.tick_params(labelsize=6)
    ax.grid(True, alpha=0.3)
    if i == 0:
        ax.legend(fontsize=6)
plt.suptitle('Prédictions vs Réel — 140 Capteurs (SVM)', fontsize=16, fontweight='bold', y=1.001)
plt.tight_layout()
plt.show()

In [ ]:
cm_svm = confusion_matrix(y_tb_svm, y_pb_svm)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm_svm, annot=True, fmt='d', cmap='Greens', ax=ax,
            xticklabels=['Normal', 'Congestion'], yticklabels=['Normal', 'Congestion'])
ax.set_xlabel('Prédit'); ax.set_ylabel('Réel')
ax.set_title('Matrice de Confusion — Détection de Congestion (SVM)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:


residuals_svm = (y_true_svm - y_pred_svm).flatten()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
ax.hist(residuals_svm, bins=80, color='#2ecc71', edgecolor='white', alpha=0.8, density=True)
ax.axvline(0, color='red', ls='--', lw=2, label=f'Moyenne = {residuals_svm.mean():.4f}')
ax.set_xlabel('Résidu (Réel − Prédit)')
ax.set_ylabel('Densité')
ax.set_title('Distribution des Résidus — SVM', fontsize=13, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)


ax = axes[1]
sample_idx = np.random.choice(len(y_true_svm.flatten()), size=5000, replace=False)
ax.scatter(y_true_svm.flatten()[sample_idx], y_pred_svm.flatten()[sample_idx],
           alpha=0.3, s=5, color='#2ecc71')
lims = [0, 1]
ax.plot(lims, lims, 'r--', lw=2, label='Prédiction Parfaite')
ax.set_xlabel('Valeur Réelle'); ax.set_ylabel('Valeur Prédite')
ax.set_title('Réel vs Prédit — SVM', fontsize=13, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)
ax.set_xlim(lims); ax.set_ylim(lims)

plt.tight_layout()
plt.show()


In [ ]:

mse_per_sensor_svm = np.mean((y_true_svm - y_pred_svm) ** 2, axis=0)
fig, ax = plt.subplots(figsize=(16, 6))
ax.bar(range(N), mse_per_sensor_svm,
       color=plt.cm.Greens(mse_per_sensor_svm / mse_per_sensor_svm.max()), edgecolor='none')
ax.axhline(mse_per_sensor_svm.mean(), color='darkgreen', ls='--', lw=2,
           label=f'MSE Moyenne = {mse_per_sensor_svm.mean():.6f}')
ax.set_xlabel('Capteur'); ax.set_ylabel('MSE')
ax.set_title('Erreur MSE par Capteur — Performance Spatiale (SVM)', fontsize=14, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:

def get_fourier_series(t, p=288, n=2):
    x = 2 * np.pi * t / p
    fourier = []
    for i in range(1, n + 1):
        fourier.append(np.sin(i * x))
        fourier.append(np.cos(i * x))
    return np.column_stack(fourier)

N_TEST = len(y_test)
t_train = np.arange(split_idx)
t_test = np.arange(split_idx, split_idx + N_TEST)


exog_train = get_fourier_series(t_train)
exog_test = get_fourier_series(t_test)

start = time.time()

y_pred_arima = np.zeros((N_TEST, N))

for i in range(N):
    try:
       
        model_arima = SARIMAX(train_data[:, i], exog=exog_train, order=(2, 0, 0))
        results = model_arima.fit(disp=False)
        forecast = results.forecast(steps=N_TEST, exog=exog_test)
        y_pred_arima[:, i] = np.clip(forecast, 0, 1)
    except:
        y_pred_arima[:, i] = train_data[:, i].mean()
    
    if (i + 1) % 20 == 0 or i == N - 1:
        print(f"  Capteur {i+1}/{N} terminé")

end = time.time()   

temps_arima = end - start
print("Temps d'execution ARIMA :", temps_arima, "secondes")

y_true_arima = test_data[SEQ_LENGTH : SEQ_LENGTH + N_TEST]
mse_arima = mean_squared_error(y_true_arima, y_pred_arima)
rmse_arima = np.sqrt(mse_arima)
mae_arima = mean_absolute_error(y_true_arima, y_pred_arima)
r2_arima   = r2_score(y_true_arima.flatten(), y_pred_arima.flatten())

y_tb_arima = (y_true_arima < CONGESTION_THRESHOLD).astype(int).flatten()
y_pb_arima = (y_pred_arima < CONGESTION_THRESHOLD).astype(int).flatten()
prec_arima = precision_score(y_tb_arima, y_pb_arima, zero_division=0)
rec_arima = recall_score(y_tb_arima, y_pb_arima, zero_division=0)
f1_arima = f1_score(y_tb_arima, y_pb_arima, zero_division=0)

print('\n' + '='*50)
print('     RÉSULTATS ARIMA (Fourier Seasonality)')
print('='*50)
print(f'  MSE       : {mse_arima:.6f}')
print(f'  RMSE      : {rmse_arima:.6f}')
print(f" R²        : {r2_arima:.6f}") 
print(f'  MAE       : {mae_arima:.6f}')
print(f'  Precision : {prec_arima:.4f}')
print(f'  Recall    : {rec_arima:.4f}')
print(f'  F1-Score  : {f1_arima:.4f}')
print('='*50)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, s in zip(axes.flatten(), sample_sensors):
    ax.plot(y_true_arima[:200, s], label='Réel', lw=1.5, color='steelblue')
    ax.plot(y_pred_arima[:200, s], label='ARIMA (Fourier)', lw=1.5, color='purple', linestyle='--')
    ax.set_title(f'Capteur {s}', fontweight='bold')
    ax.legend(); ax.grid(True, alpha=0.3)
plt.suptitle('ARIMA avec Saisonnalité 24h — Prédictions vs Réel', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:

fig, axes = plt.subplots(20, 7, figsize=(35, 60))
for i, ax in enumerate(axes.flatten()):
    ax.plot(y_true_arima[:200, i], lw=1, alpha=0.8, color='steelblue', label='Réel')
    ax.plot(y_pred_arima[:200, i], lw=1, alpha=0.8, color='purple', linestyle='--', label='ARIMA')
    ax.set_title(f'Capteur {i}', fontsize=8, fontweight='bold')
    ax.set_xlabel('Timestamp', fontsize=6)
    ax.set_ylabel('Vitesse', fontsize=6)
    ax.tick_params(labelsize=6)
    ax.grid(True, alpha=0.3)
    if i == 0:
        ax.legend(fontsize=6)
plt.suptitle('Prédictions vs Réel — 140 Capteurs (ARIMA)', fontsize=16, fontweight='bold', y=1.001)
plt.tight_layout()
plt.show()

In [ ]:
cm_arima = confusion_matrix(y_tb_arima, y_pb_arima)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm_arima, annot=True, fmt='d', cmap='Purples', ax=ax,
            xticklabels=['Normal', 'Congestion'], yticklabels=['Normal', 'Congestion'])
ax.set_xlabel('Prédit'); ax.set_ylabel('Réel')
ax.set_title('Matrice de Confusion — Détection de Congestion (ARIMA)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:

residuals_arima = (y_true_arima - y_pred_arima).flatten()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))


ax = axes[0]
ax.hist(residuals_arima, bins=80, color='#9b59b6', edgecolor='white', alpha=0.8, density=True)
ax.axvline(0, color='red', ls='--', lw=2, label=f'Moyenne = {residuals_arima.mean():.4f}')
ax.set_xlabel('Résidu (Réel − Prédit)')
ax.set_ylabel('Densité')
ax.set_title('Distribution des Résidus — ARIMA', fontsize=13, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)


ax = axes[1]
sample_idx = np.random.choice(len(y_true_arima.flatten()), size=5000, replace=False)
ax.scatter(y_true_arima.flatten()[sample_idx], y_pred_arima.flatten()[sample_idx],
           alpha=0.3, s=5, color='#9b59b6')
lims = [0, 1]
ax.plot(lims, lims, 'r--', lw=2, label='Prédiction Parfaite')
ax.set_xlabel('Valeur Réelle'); ax.set_ylabel('Valeur Prédite')
ax.set_title('Réel vs Prédit — ARIMA', fontsize=13, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)
ax.set_xlim(lims); ax.set_ylim(lims)

plt.tight_layout()
plt.show()


In [ ]:

mse_per_sensor_arima = np.mean((y_true_arima - y_pred_arima) ** 2, axis=0)
fig, ax = plt.subplots(figsize=(16, 6))
ax.bar(range(N), mse_per_sensor_arima,
       color=plt.cm.Purples(mse_per_sensor_arima / mse_per_sensor_arima.max()), edgecolor='none')
ax.axhline(mse_per_sensor_arima.mean(), color='purple', ls='--', lw=2,
           label=f'MSE Moyenne = {mse_per_sensor_arima.mean():.6f}')
ax.set_xlabel('Capteur'); ax.set_ylabel('MSE')
ax.set_title('Erreur MSE par Capteur — Performance Spatiale (ARIMA)', fontsize=14, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os, warnings
from datetime import datetime, timedelta
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error,
    precision_score, recall_score, f1_score,
    confusion_matrix, r2_score
)
import tensorflow as tf
from keras.models import Sequential
from keras.layers import LSTM as LSTM_Layer, Dense, Dropout, Input
from keras.callbacks import EarlyStopping, ReduceLROnPlateau

warnings.filterwarnings('ignore')
tf.get_logger().setLevel('ERROR')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

# ── Chargement des données ──
DATA_DIR = r"C:\Users\Hp\Desktop\Atelier\Atelier\Pems07\PEMS07"
npz_path = os.path.join(DATA_DIR, "PEMS07.npz")
npz_data = np.load(npz_path)
data = npz_data['data'] if 'data' in npz_data else npz_data[list(npz_data.keys())[0]]
if data.ndim == 3:
    data = data[:, :, 0]
T, N = data.shape

# ── Normalisation ──
scaler = MinMaxScaler(feature_range=(0, 1))
data_norm = scaler.fit_transform(data)

# ── Paramètres ──
SEQ_LENGTH = 12
TEST_RATIO = 0.20
CONGESTION_THRESHOLD = 0.4

split_idx = int(T * (1 - TEST_RATIO))
train_data = data_norm[:split_idx]
test_data = data_norm[split_idx:]

def create_sequences(data, seq_len):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i+seq_len])
        y.append(data[i+seq_len])
    return np.array(X), np.array(y)

X_train, y_train = create_sequences(train_data, SEQ_LENGTH)
X_test, y_test = create_sequences(test_data, SEQ_LENGTH)

sample_sensors = [0, 35, 70, 105]

print(f'Données rechargées : {T} timestamps × {N} capteurs')
print(f'X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'X_test : {X_test.shape},  y_test : {y_test.shape}')


In [ ]:

LSTM_UNITS = 64
EPOCHS_LSTM = 30
BATCH_SIZE_LSTM = 64

lstm_model = Sequential([
    Input(shape=(SEQ_LENGTH, N)),
    LSTM_Layer(LSTM_UNITS, activation='tanh', return_sequences=True),
    Dropout(0.2),
    LSTM_Layer(LSTM_UNITS // 2),
    Dropout(0.2),
    Dense(N)
])

lstm_model.compile(optimizer='adam', loss='mse', metrics=['mae'])
lstm_model.summary()


In [ ]:
callbacks_lstm = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
]
start = time.time()

history_lstm = lstm_model.fit(
    X_train, y_train,
    epochs=EPOCHS_LSTM,
    batch_size=BATCH_SIZE_LSTM,
    validation_split=0.1,
    callbacks=callbacks_lstm,
    verbose=1
)
end = time.time()

temps_lstm = end - start
print("Temps d'entrainement LSTM :", temps_lstm, "secondes")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(history_lstm.history['loss'], label='Train Loss', lw=2)
ax.plot(history_lstm.history['val_loss'], label='Validation Loss', lw=2)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss (MSE)')
ax.set_title('Courbes de Perte — LSTM', fontsize=14, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
y_pred_lstm = lstm_model.predict(X_test, verbose=0)

mse_lstm = mean_squared_error(y_test, y_pred_lstm)
rmse_lstm = np.sqrt(mse_lstm)
mae_lstm = np.mean(np.abs(y_test - y_pred_lstm))
r2_lstm = r2_score(y_test.flatten(), y_pred_lstm.flatten())

y_tb_lstm = (y_test < CONGESTION_THRESHOLD).astype(int).flatten()
y_pb_lstm = (y_pred_lstm < CONGESTION_THRESHOLD).astype(int).flatten()

prec_lstm = precision_score(y_tb_lstm, y_pb_lstm, average='binary', zero_division=0)
rec_lstm = recall_score(y_tb_lstm, y_pb_lstm, average='binary', zero_division=0)
f1_lstm = f1_score(y_tb_lstm, y_pb_lstm, average='binary', zero_division=0)

print('╔' + '═'*45 + '╗')
print('║        RÉSULTATS DU MODÈLE LSTM            ║')
print('╠' + '═'*45 + '╣')
print(f'║  MSE       : {mse_lstm:.6f}')
print(f'║  RMSE      : {rmse_lstm:.6f}')
print(f'║  MAE       : {mae_lstm:.6f}')
print(f'║  R²        : {r2_lstm:.6f}')
print('╠' + '═'*45 + '╣')
print(f'║  Precision : {prec_lstm:.4f}')
print(f'║  Recall    : {rec_lstm:.4f}')
print(f'║  F1-Score  : {f1_lstm:.4f}')
print('╚' + '═'*45 + '╝')


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, sensor in zip(axes.flatten(), sample_sensors):
    ax.plot(y_test[:200, sensor], label='Réel', lw=1.5, alpha=0.8, color='steelblue')
    ax.plot(y_pred_lstm[:200, sensor], label='LSTM', lw=1.5, alpha=0.8, color='crimson')
    ax.set_title(f'Capteur {sensor}', fontweight='bold')
    ax.set_xlabel('Timestamp'); ax.set_ylabel('Vitesse (norm.)')
    ax.legend(); ax.grid(True, alpha=0.3)
plt.suptitle('LSTM — Prédictions vs Réel (Échantillon)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:

fig, axes = plt.subplots(20, 7, figsize=(35, 60))
for i, ax in enumerate(axes.flatten()):
    ax.plot(y_test[:200, i], lw=1, alpha=0.8, color='steelblue', label='Réel')
    ax.plot(y_pred_lstm[:200, i], lw=1, alpha=0.8, color='crimson', label='LSTM')
    ax.set_title(f'Capteur {i}', fontsize=8, fontweight='bold')
    ax.set_xlabel('Timestamp', fontsize=6)
    ax.set_ylabel('Vitesse', fontsize=6)
    ax.tick_params(labelsize=6)
    ax.grid(True, alpha=0.3)
    if i == 0:
        ax.legend(fontsize=6)
plt.suptitle('Prédictions vs Réel — 140 Capteurs (LSTM)', fontsize=16, fontweight='bold', y=1.001)
plt.tight_layout()
plt.show()


In [ ]:
cm_lstm = confusion_matrix(y_tb_lstm, y_pb_lstm)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm_lstm, annot=True, fmt='d', cmap='Reds', ax=ax,
            xticklabels=['Normal', 'Congestion'], yticklabels=['Normal', 'Congestion'])
ax.set_xlabel('Prédit'); ax.set_ylabel('Réel')
ax.set_title('Matrice de Confusion — Détection de Congestion (LSTM)', fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
mse_per_sensor_lstm = np.mean((y_test - y_pred_lstm) ** 2, axis=0)
fig, ax = plt.subplots(figsize=(16, 6))
ax.bar(range(N), mse_per_sensor_lstm,
       color=plt.cm.Reds(mse_per_sensor_lstm / mse_per_sensor_lstm.max()), edgecolor='none')
ax.axhline(mse_per_sensor_lstm.mean(), color='red', ls='--', lw=2,
           label=f'MSE Moyenne = {mse_per_sensor_lstm.mean():.6f}')
ax.set_xlabel('Capteur'); ax.set_ylabel('MSE')
ax.set_title('Erreur MSE par Capteur — Performance Spatiale (LSTM)', fontsize=14, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
class TrainableGCN(tf.keras.layers.Layer):
    def __init__(self, units, **kwargs):
        super().__init__(**kwargs)
        self.units = units

    def build(self, input_shape):
        self.N = input_shape[1]
        self.A = self.add_weight(shape=(self.N, self.N), initializer='glorot_uniform', trainable=True)
        self.W = self.add_weight(shape=(input_shape[-1], self.units), initializer='glorot_uniform', trainable=True)

    def call(self, inputs):
        A = tf.nn.softmax(self.A, axis=1)
        x = tf.einsum('ij,bjk->bik', A, inputs)
        x = tf.matmul(x, self.W)
        return tf.nn.relu(x)

inputs_gcn = Input(shape=(SEQ_LENGTH, N))
x = tf.keras.layers.Permute((2, 1))(inputs_gcn)
x = TrainableGCN(64)(x)
x = TrainableGCN(32)(x)
x = tf.keras.layers.Flatten()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.2)(x)
outputs_gcn = Dense(N)(x)

gcn_model = tf.keras.Model(inputs_gcn, outputs_gcn)
gcn_model.compile(optimizer='adam', loss='mse', metrics=['mae'])
gcn_model.summary()


In [ ]:
start = time.time()
history_gcn = gcn_model.fit(
    X_train, y_train,
    epochs=EPOCHS_LSTM,
    batch_size=BATCH_SIZE_LSTM,
    validation_split=0.1,
    callbacks=callbacks_lstm,
    verbose=1
)
end = time.time()
temps_gcn = end - start
print("Temps d'entrainement GCN :", temps_gcn, "secondes")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(history_gcn.history['loss'], label='Train Loss', lw=2)
ax.plot(history_gcn.history['val_loss'], label='Validation Loss', lw=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss (MSE)')
ax.set_title('Courbes de Perte — GCN', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
y_pred_gcn = gcn_model.predict(X_test, verbose=0)

mse_gcn = mean_squared_error(y_test, y_pred_gcn)
rmse_gcn = np.sqrt(mse_gcn)
mae_gcn = np.mean(np.abs(y_test - y_pred_gcn))
r2_gcn = r2_score(y_test.flatten(), y_pred_gcn.flatten())

y_tb_gcn = (y_test < CONGESTION_THRESHOLD).astype(int).flatten()
y_pb_gcn = (y_pred_gcn < CONGESTION_THRESHOLD).astype(int).flatten()

prec_gcn = precision_score(y_tb_gcn, y_pb_gcn, average='binary', zero_division=0)
rec_gcn = recall_score(y_tb_gcn, y_pb_gcn, average='binary', zero_division=0)
f1_gcn = f1_score(y_tb_gcn, y_pb_gcn, average='binary', zero_division=0)

print('╔' + '═'*45 + '╗')
print('║        RÉSULTATS DU MODÈLE GCN             ║')
print('╠' + '═'*45 + '╣')
print(f'║  MSE       : {mse_gcn:.6f}')
print(f'║  RMSE      : {rmse_gcn:.6f}')
print(f'║  MAE       : {mae_gcn:.6f}')
print(f'║  R²        : {r2_gcn:.6f}')
print('╠' + '═'*45 + '╣')
print(f'║  Precision : {prec_gcn:.4f}')
print(f'║  Recall    : {rec_gcn:.4f}')
print(f'║  F1-Score  : {f1_gcn:.4f}')
print('╚' + '═'*45 + '╝')


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, sensor in zip(axes.flatten(), sample_sensors):
    ax.plot(y_test[:200, sensor],     label='Réel',  lw=1.5, alpha=0.8, color='steelblue')
    ax.plot(y_pred_gcn[:200, sensor], label='GCN',   lw=1.5, alpha=0.8, color='darkorange')
    ax.set_title(f'Capteur {sensor}', fontweight='bold')
    ax.set_xlabel('Timestamp')
    ax.set_ylabel('Vitesse (norm.)')
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.suptitle('GCN — Prédictions vs Réel (Échantillon)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(20, 7, figsize=(35, 60))
for i, ax in enumerate(axes.flatten()):
    ax.plot(y_test[:200, i],     lw=1, alpha=0.8, color='steelblue',  label='Réel')
    ax.plot(y_pred_gcn[:200, i], lw=1, alpha=0.8, color='darkorange', label='GCN')
    ax.set_title(f'Capteur {i}', fontsize=8, fontweight='bold')
    ax.set_xlabel('Timestamp', fontsize=6)
    ax.set_ylabel('Vitesse',   fontsize=6)
    ax.tick_params(labelsize=6)
    ax.grid(True, alpha=0.3)
    if i == 0:
        ax.legend(fontsize=6)
plt.suptitle('Prédictions vs Réel — 140 Capteurs (GCN)',
             fontsize=16, fontweight='bold', y=1.001)
plt.tight_layout()
plt.show()

In [ ]:
cm_gcn = confusion_matrix(y_tb_gcn, y_pb_gcn)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm_gcn, annot=True, fmt='d', cmap='Oranges', ax=ax,
            xticklabels=['Normal', 'Congestion'],
            yticklabels=['Normal', 'Congestion'])
ax.set_xlabel('Prédit')
ax.set_ylabel('Réel')
ax.set_title('Matrice de Confusion — Détection de Congestion (GCN)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
mse_per_sensor_gcn = np.mean((y_test - y_pred_gcn) ** 2, axis=0)
 
fig, ax = plt.subplots(figsize=(16, 6))
ax.bar(range(N), mse_per_sensor_gcn,
       color=plt.cm.Oranges(mse_per_sensor_gcn / mse_per_sensor_gcn.max()),
       edgecolor='none')
ax.axhline(mse_per_sensor_gcn.mean(), color='darkorange', ls='--', lw=2,
           label=f'MSE Moyenne = {mse_per_sensor_gcn.mean():.6f}')
ax.set_xlabel('Capteur')
ax.set_ylabel('MSE')
ax.set_title('Erreur MSE par Capteur — Performance Spatiale (GCN)',
             fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:

tableau_all = pd.DataFrame({
    'Modèle': ['SVM ', 'SimpleRNN', 'ARIMA', 'LSTM', 'GCN'],
    'MSE': [mse_svm, mse, mse_arima, mse_lstm, mse_gcn],
    'RMSE': [rmse_svm, rmse, rmse_arima, rmse_lstm, rmse_gcn],
    'MAE': [mae_svm, mae, mae_arima, mae_lstm, mae_gcn],
    'R²': [r2_svm, r2, r2_arima, r2_lstm, r2_gcn],
    'Precision': [prec_svm, precision, prec_arima, prec_lstm, prec_gcn],
    'Recall': [rec_svm, recall, rec_arima, rec_lstm, rec_gcn],
    'F1-Score': [f1_svm, f1, f1_arima, f1_lstm, f1_gcn]
}).set_index('Modèle')

print('═'*85)
print('         TABLEAU COMPARATIF — TOUS LES MODÈLES (PEMS07)')
print('═'*85)
print(tableau_all.round(6).to_string())
print('═'*85)


In [ ]:
modeles_all = ['SVM', 'SimpleRNN', 'ARIMA', 'LSTM', 'GCN']
colors_all = ['#2ecc71', '#3498db', '#9b59b6', '#e74c3c', '#f39c12']

fig, axes = plt.subplots(1, 4, figsize=(24, 6))
metrics_all = {
    'RMSE': [rmse_svm, rmse, rmse_arima, rmse_lstm, rmse_gcn],
    'MAE': [mae_svm, mae, mae_arima, mae_lstm, mae_gcn],
    'R²': [r2_svm, r2, r2_arima, r2_lstm, r2_gcn],
    'F1-Score': [f1_svm, f1, f1_arima, f1_lstm, f1_gcn],
}
for ax, (name, vals) in zip(axes, metrics_all.items()):
    bars = ax.bar(modeles_all, vals, color=colors_all, edgecolor='white', width=0.5)
    ax.set_title(name, fontsize=13, fontweight='bold')
    ax.set_ylabel(name); ax.grid(True, alpha=0.3, axis='y')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.001,
                f'{val:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
plt.suptitle('Comparaison Complète : SVM vs SimpleRNN vs ARIMA vs LSTM vs GCN (PEMS07)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:

temps = {
    "SVM": temps_svm,
    "ARIMA": temps_arima,
    "LSTM": temps_lstm,
    "RNN": temps_rnn,
    "GCN": temps_gcn
}

df_temps = pd.DataFrame(list(temps.items()), columns=["Modèle", "Temps (s)"])
df_temps